LIBRERIAS

In [30]:
# ManipulaciOn de datos
import pandas as pd
import numpy as np
from datetime import datetime

# VisualizaciOn
import matplotlib.pyplot as plt

# Machine Learning
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

# Metricas
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report)

# Ignorar warnings
import warnings
warnings.filterwarnings("ignore")

ARCHIVOS UTILIZADOS

In [31]:
solicitudes_train = pd.read_csv("solicitudes_train.csv")
comportamiento_bancario = pd.read_csv("comportamiento_bancario.csv")
solicitudes_test = pd.read_csv("solicitudes_test.csv")

UNIFICACION DE DATOS ENTRE SOLICITUDES_TRAIN Y COMPORTAMIENTO BANCARIO PARA CREAR UN DATASET_FINAL_TRAIN

In [32]:
# Conversion de campos de fecha a datetime
solicitudes_train["Fecha_Solicitud"] = pd.to_datetime(solicitudes_train["Fecha_Solicitud"])
comportamiento_bancario["Mes_Referencia"] = pd.to_datetime(comportamiento_bancario["Mes_Referencia"])

# Unificacion de los datos de las solicitudes con el comportamiento bancario
datos_combinados_train = solicitudes_train.merge(comportamiento_bancario, on="ID_Usuario", how="left")

# Filtro para prevenir leakage (solo es valido el comportamiento bancario de antes de la solicitud)
datos_combinados_train = datos_combinados_train[datos_combinados_train["Mes_Referencia"] < datos_combinados_train["Fecha_Solicitud"]]

# Feature engineering (promedio del saldo historico, promedio de atrasos historicos, maximo de atrasos historicos y hace cuanto tiene la cuenta)
historico_train = (datos_combinados_train.groupby("ID_Solicitud").agg(
        Saldo_Historico_Promedio=("Saldo_Promedio_Mensual", "mean"),
        Atrasos_Historicos_Promedio=("Dias_Atraso_Otras_Deudas", "mean"),
        Atrasos_Historicos_Max=("Dias_Atraso_Otras_Deudas", "max"),
        Meses_Historial_Cuenta=("Mes_Referencia", "count"),
    ).reset_index())

# Dataset definitivo combinando las solicitudes, el comportamiento bancario y las nuevas variables
datos_final_train = solicitudes_train.merge(historico_train, on="ID_Solicitud", how="left")
datos_final_train.columns

Index(['ID_Solicitud', 'ID_Usuario', 'Monto_Solicitado', 'Tasa_Interes',
       'Ingreso_Mensual', 'Motivo_Prestamo', 'Fecha_Solicitud', 'Estado_Final',
       'Saldo_Historico_Promedio', 'Atrasos_Historicos_Promedio',
       'Atrasos_Historicos_Max', 'Meses_Historial_Cuenta'],
      dtype='object')

SEPARACION DEL DATASET EN ENTRADAS/SALIDAS Y DIVISION EN TRAIN/VALIDATION

In [33]:
# Separarcion de entradas y salidas (antes del split, para que train y validation tengan las mismas variables)
# Drop de columnas no desesadas o que no tienen relevancia
x = datos_final_train.drop(
    columns=[
        "Estado_Final",
        "ID_Solicitud",
        "ID_Usuario",
        "Fecha_Solicitud",
        "Motivo_Prestamo",
    ]
)
y = datos_final_train["Estado_Final"]

# Split en entrenamiento y validacion
x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Visualizacion de Resultados
x_train.sample(5)

,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Saldo_Historico_Promedio,Atrasos_Historicos_Promedio,Atrasos_Historicos_Max,Meses_Historial_Cuenta
4210,10800,0.2780,2992,4346.386000,2.800000,20,15
3096,45900,0.2447,11494,9237.031333,15.600000,47,15
27297,15700,0.2623,2768,34.880435,18.217391,54,23
28222,5800,0.2862,1734,1342.784667,8.933333,40,15
3623,17500,0.2882,8099,10183.985556,9.777778,56,18


PIPELINE (STANDARSCALER EN LAS VARIABLES NUMERICAS)

In [37]:
# Variables numericas
numeric_features = [
    "Monto_Solicitado",
    "Tasa_Interes",
    "Ingreso_Mensual",
    "Saldo_Historico_Promedio",
    "Atrasos_Historicos_Promedio",
    "Atrasos_Historicos_Max",
    "Meses_Historial_Cuenta",
]

# Pipeline numérico
numeric_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
    ]
)

# Preprocesador
preprocessor = ColumnTransformer(transformers=[("num", numeric_pipeline, numeric_features)])

# Pipeline completo
modelo_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("clf", LogisticRegression(class_weight="balanced", random_state=42),),
    ]
)

# Entrenar todo de un solo golpe
modelo_pipeline.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Monto_Solicitado',
                                                   'Tasa_Interes',
                                                   'Ingreso_Mensual',
                                                   'Saldo_Historico_Promedio',
                                                   'Atrasos_Historicos_Promedio',
                                                   'Atrasos_Historicos_Max',
                                                   'Meses_Historial_Cuenta'])])),
                ('clf',
                 LogisticRegression(class_weight='balanced', random_state=42))])

PRUEBA DE PREDICCIONES BASICAS SOBRE EL DATASET DE VALIDACION

In [38]:
y_pred = modelo_pipeline.predict(x_val)

# Visualizacion de predicciones junto con el valor real
resultado = x_val.copy()
resultado["Estado_Real"] = y_val
resultado["Estado_Predicho"] = y_pred

# Visualizacion de Resultados
resultado[["Monto_Solicitado", "Tasa_Interes", "Estado_Real", "Estado_Predicho"]].sample(5)

,Monto_Solicitado,Tasa_Interes,Estado_Real,Estado_Predicho
30384,9700,0.2402,1,1
27827,52600,0.3524,1,1
30230,19200,0.2261,1,1
14199,21700,0.2611,1,1
29053,39300,0.2587,1,1


METRICAS VISTAS (EN ESTE CASO PRACTICAMENTE NO TIENEN UTILIDAD REAL)

In [39]:
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.38      0.74      0.50      1034
           1       0.94      0.77      0.85      5450

    accuracy                           0.77      6484
   macro avg       0.66      0.75      0.68      6484
weighted avg       0.85      0.77      0.79      6484



ENTONCES ENCARAMOS POR OTRO LADO \
PRIMERO, EN VEZ DE HACER PREDICT, CALCULAMOS LAS PROBABILIDADES DE PAGO

---
SE UTILIZA predict_proba QUE CALCULA LA PROBABILIDAD DE QUE SE PAGUE O NO EL PRESTAMO CON LA INFORMACION APRENDIDA AL HACER FIT

In [40]:
validacion_analisis = x_val.copy()
validacion_analisis["Estado_Real"] = y_val

# Probabilidades predichas de que se pague o no el prestamo. Se guarda la probabilidad de que pague.
validacion_analisis["Prob_Pago"] = modelo_pipeline.predict_proba(x_val)[:, 1]

# Visualizacion de Resultados
validacion_analisis.head(5)

,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Saldo_Historico_Promedio,Atrasos_Historicos_Promedio,Atrasos_Historicos_Max,Meses_Historial_Cuenta,Estado_Real,Prob_Pago
28113,3400,0.2328,2985,1396.517692,11.692308,48,13,1,0.552487
30975,4300,0.2485,1096,850.577500,20.650000,65,20,1,0.187200
11314,10700,0.2807,3450,3160.016250,10.812500,40,16,1,0.489578
18808,3100,0.2410,2789,5236.565238,10.476190,59,21,1,0.745604
24802,5900,0.2304,3724,7228.668333,1.333333,5,18,1,0.894201


**CALCULO DE ESPERANZA Y ROI**:\
**Valor Esperado (Esperanza Matemática)**: dice cuántos dólares se esperan ganar en total con un préstamo determinado.\
**ROI Esperado**: dice qué tan eficiente es ese préstamo para exprimir al máximo el presupuesto limitado de $2M USD.

In [41]:
# Probabilidad de default
prob_default = 1 - validacion_analisis["Prob_Pago"]

# Ganancia potencial (si paga) y Pérdida potencial (si hace default)
ganancia_potencial = validacion_analisis["Monto_Solicitado"] * validacion_analisis["Tasa_Interes"]
perdida_potencial = -validacion_analisis["Monto_Solicitado"]

# Cálculo del Valor Esperado
validacion_analisis["Valor_Esperado"] = (validacion_analisis["Prob_Pago"] * ganancia_potencial) + (prob_default * perdida_potencial)

# Cálculo del ROI Esperado (Retorno por cada $1 invertido)
validacion_analisis["ROI_Esperado"] = validacion_analisis["Valor_Esperado"] / validacion_analisis["Monto_Solicitado"]

# Visualizacion de Resultados
validacion_analisis[["Monto_Solicitado", "Tasa_Interes", "Atrasos_Historicos_Promedio", "Atrasos_Historicos_Max", "Estado_Real", "Prob_Pago", "Valor_Esperado", "ROI_Esperado"]].sample(10)

,Monto_Solicitado,Tasa_Interes,Atrasos_Historicos_Promedio,Atrasos_Historicos_Max,Estado_Real,Prob_Pago,Valor_Esperado,ROI_Esperado
5282,23100,0.3042,12.142857,57,1,0.355652,-12385.266748,-0.536159
12260,20400,0.2872,25.833333,63,0,0.038616,-19385.991963,-0.950294
29172,6200,0.2542,1.153846,6,1,0.950860,1193.924018,0.192568
5854,4000,0.2202,2.235294,5,1,0.802041,-85.397694,-0.021349
28109,8300,0.2725,13.500000,45,0,0.500259,-3016.385762,-0.363420
24941,10300,0.2880,19.631579,63,1,0.161775,-8153.826258,-0.791634
19972,35900,0.2965,0.954545,4,1,0.875682,4858.052755,0.135322
1151,13200,0.2895,3.250000,10,1,0.665662,-1869.492949,-0.141628
28281,5500,0.2359,1.916667,5,1,0.852469,294.618428,0.053567
15351,7700,0.2486,9.066667,15,1,0.474078,-3142.108446,-0.408066


ORDEN SEGUN RENTABILIDAD

In [42]:
# Conservar solo las solicitudes donde no haya perdida. Sirven para que si hay pocas solicitudes rentables, no siga aprobandolas hasta agotar el dinero.
rentables = validacion_analisis[validacion_analisis["Valor_Esperado"] > 0].copy()

# Ordenar de mayor a menor ROI Esperado (los préstamos más eficientes primero)
rentables = rentables.sort_values(by="ROI_Esperado", ascending=False)

# Visualizacion de Resultados
rentables[["Monto_Solicitado", "Tasa_Interes", "Atrasos_Historicos_Promedio", "Atrasos_Historicos_Max", "Prob_Pago", "Valor_Esperado", "ROI_Esperado"]].head(10)

,Monto_Solicitado,Tasa_Interes,Atrasos_Historicos_Promedio,Atrasos_Historicos_Max,Prob_Pago,Valor_Esperado,ROI_Esperado
19979,20400,0.3151,2.428571,23,0.997160,6351.860118,0.311366
11382,70000,0.3227,2.954545,54,0.987665,21446.935003,0.306385
10183,70000,0.2977,5.090909,49,0.996650,20534.691385,0.293353
28394,55100,0.3230,0.904762,8,0.976352,16073.428347,0.291714
27605,10300,0.3059,3.125000,34,0.983504,2928.886567,0.284358
20583,44500,0.3076,6.117647,56,0.980494,12553.173601,0.282094
30598,70000,0.2803,2.111111,18,0.996101,19271.591669,0.275308
20450,18200,0.2958,4.857143,28,0.981649,4950.772541,0.272020
12904,38800,0.2934,6.304348,50,0.982100,10485.614992,0.270248
19604,21100,0.2933,3.954545,56,0.981493,5683.586630,0.269364


QUEDARSE CON LAS SOLICITUDES MAS RENTABLES HASTA AGOTAR PRESUPUESTO

In [43]:
# Presupuesto maximo
PRESUPUESTO_MAX = 2000000

# Suma acumulada del capital solicitado en la cartera ordenada
rentables["Monto_Acumulado"] = rentables["Monto_Solicitado"].cumsum()

rentables[["Monto_Solicitado", "Tasa_Interes", "Estado_Real", "Prob_Pago", "Valor_Esperado", "ROI_Esperado", "Monto_Acumulado"]].head(10)

,Monto_Solicitado,Tasa_Interes,Estado_Real,Prob_Pago,Valor_Esperado,ROI_Esperado,Monto_Acumulado
19979,20400,0.3151,1,0.997160,6351.860118,0.311366,20400
11382,70000,0.3227,1,0.987665,21446.935003,0.306385,90400
10183,70000,0.2977,1,0.996650,20534.691385,0.293353,160400
28394,55100,0.3230,1,0.976352,16073.428347,0.291714,215500
27605,10300,0.3059,1,0.983504,2928.886567,0.284358,225800
20583,44500,0.3076,1,0.980494,12553.173601,0.282094,270300
30598,70000,0.2803,1,0.996101,19271.591669,0.275308,340300
20450,18200,0.2958,1,0.981649,4950.772541,0.272020,358500
12904,38800,0.2934,1,0.982100,10485.614992,0.270248,397300
19604,21100,0.2933,1,0.981493,5683.586630,0.269364,418400


In [44]:
aprobados = rentables[rentables["Monto_Acumulado"] <= PRESUPUESTO_MAX]

monto_total_prestado = aprobados["Monto_Solicitado"].sum()

print("Total préstamos aprobados:", len(aprobados))
print("Monto total otorgado:", monto_total_prestado)

aprobados[["Monto_Solicitado", "Tasa_Interes", "Estado_Real", "Prob_Pago", "Valor_Esperado", "ROI_Esperado", "Monto_Acumulado"]].head(67)

Total préstamos aprobados: 67
Monto total otorgado: 1986700


,Monto_Solicitado,Tasa_Interes,Estado_Real,Prob_Pago,Valor_Esperado,ROI_Esperado,Monto_Acumulado
19979,20400,0.3151,1,0.997160,6351.860118,0.311366,20400
11382,70000,0.3227,1,0.987665,21446.935003,0.306385,90400
10183,70000,0.2977,1,0.996650,20534.691385,0.293353,160400
28394,55100,0.3230,1,0.976352,16073.428347,0.291714,215500
27605,10300,0.3059,1,0.983504,2928.886567,0.284358,225800
...,...,...,...,...,...,...,...
15711,11300,0.2405,1,0.991231,2594.723265,0.229622,1887400
4745,17700,0.2359,1,0.994311,4050.988184,0.228869,1905100
11731,13400,0.2517,1,0.981540,3063.161478,0.228594,1918500
26062,30800,0.2830,1,0.956797,7009.182776,0.227571,1949300


YA TENEMOS LO QUE QUEREMOS, AHORA REPETIMOS TODO EL PROCESO CON EL DATASET DE TEST

In [45]:
# Unir solicitudes_test con el histórico bancario
datos_combinados_test = solicitudes_test.merge(comportamiento_bancario, on="ID_Usuario", how="left")
datos_combinados_test = datos_combinados_test[datos_combinados_test["Mes_Referencia"] < datos_combinados_test["Fecha_Solicitud"]]

historico_test = (    datos_combinados_test.groupby("ID_Solicitud").agg(
        Saldo_Historico_Promedio=("Saldo_Promedio_Mensual", "mean"),
        Atrasos_Historicos_Promedio=("Dias_Atraso_Otras_Deudas", "mean"),
        Atrasos_Historicos_Max=("Dias_Atraso_Otras_Deudas", "max"),
        Meses_Historial_Cuenta=("Mes_Referencia", "count"),
    ).reset_index())

test_final = solicitudes_test.merge(historico_test, on="ID_Solicitud", how="left")

# Predecir la probabilidad de pago con el modelo entrenado (modelo_pipeline)
test_final["Prob_Pago"] = modelo_pipeline.predict_proba(test_final[x_train.columns])[:, 1]

# Calcular Esperanza Matemática y ROI Esperado
# Probabilidad de default
prob_default = 1 - test_final["Prob_Pago"]

# Ganancia potencial (si paga) y Pérdida potencial (si hace default)
ganancia_potencial = (test_final["Monto_Solicitado"] * test_final["Tasa_Interes"])

perdida_potencial = -test_final["Monto_Solicitado"]

# Cálculo del Valor Esperado
test_final["Valor_Esperado"] = (test_final["Prob_Pago"] * ganancia_potencial + prob_default * perdida_potencial)

# Cálculo del ROI Esperado
test_final["ROI_Esperado"] = (test_final["Valor_Esperado"] / test_final["Monto_Solicitado"])

# Filtrar por Valor_Esperado > 0 y ordenar por ROI
candidatos_test = test_final[test_final["Valor_Esperado"] > 0].copy()
candidatos_test = candidatos_test.sort_values(by="ROI_Esperado", ascending=False)

# Aplicar tope de presupuesto
PRESUPUESTO_MAX = 2000000
candidatos_test["Monto_Acumulado"] = candidatos_test["Monto_Solicitado"].cumsum()

# Crear columna Decision_Aprobacion
decisiones = []
for monto in candidatos_test["Monto_Acumulado"]:
    if monto <= PRESUPUESTO_MAX:
        decisiones.append(1)
    else:
        decisiones.append(0)
candidatos_test["Decision_Aprobacion"] = decisiones

# Generar el archivo final de entrega
entrega = test_final[["ID_Solicitud"]].merge(candidatos_test[["ID_Solicitud", "Decision_Aprobacion"]],on="ID_Solicitud",how="left").fillna(0)
entrega.to_csv("entrega_tp3.csv", index=False)